<a href="https://colab.research.google.com/github/saileshchikkam/AI_Meeting_Intelligence/blob/main/AI_Meeting_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Project: AI Meeting Intelligence — Real & Synthetic Meeting Analyzer
### What it does

### The application has two input modes:

                  AI MEETING INTELLIGENCE
                           │
              ┌────────────┴────────────┐
              │                         │
        REAL MEETING              SYNTHETIC MEETING
              │                         │
          Audio file                  Topic
              │                         │
              ▼                         ▼
           Whisper                 Llama 3.2
              │                         │
              ▼                         ▼
         Transcript            Fake Meeting Transcript
              │                         │
              └────────────┬────────────┘
                           ▼
                     Llama 3.2
                           │
                           ▼
                  Meeting Minutes
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
           Summary    Discussion    Action Items
                           │
                           ▼
                       Gradio UI

This project demonstrates an AI-powered meeting intelligence system built using
open-source models from Hugging Face.

The application supports two types of input:

1. **Real Meeting Mode** — Upload a meeting audio file and convert it into text
   using Whisper.
2. **Synthetic Meeting Mode** — Provide a meeting topic and generate an
   artificial meeting conversation using an open-source LLM.

The resulting transcript is then processed using Llama 3.2 3B Instruct to
generate structured meeting minutes.

### Output

The application generates:

- Meeting Summary
- Discussion Points
- Takeaways
- Action Items with Owners

A Gradio interface is used to provide an interactive user interface.

### Technologies

- Python
- Google Colab
- Hugging Face Transformers
- Hugging Face Pipelines
- Whisper
- Llama 3.2 3B Instruct
- Tokenization
- Chat Templates
- 4-bit Quantization
- Gradio

## 1. Environment Verification

The project uses GPU acceleration for Whisper and the quantized Llama model.

We will first verify that a CUDA-compatible GPU is available.

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Install Required Libraries

We install the libraries required for:

- Hugging Face Transformers
- 4-bit model quantization
- GPU acceleration
- Gradio interface

In [3]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.0 MB/s eta 0:00:00


## 3. Imports

We import the Hugging Face and Python components required by the project.

The main components are:

- `pipeline` for Whisper
- `AutoTokenizer` for tokenization
- `AutoModelForCausalLM` for Llama
- `BitsAndBytesConfig` for 4-bit quantization
- `TextStreamer` for streaming generated text
- `gradio` for the user interface

In [4]:
import os
import torch
import gradio as gr

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig
)

from huggingface_hub import login
from google.colab import userdata

## 4. Hugging Face Authentication

The Llama model is hosted on Hugging Face.

We will store the Hugging Face access token in Google Colab Secrets
instead of writing the token directly in the notebook.

### Before running the next cell:

1. Open the left sidebar in Colab.
2. Click the 🔑 Secrets icon.
3. Add a secret named:

HF_TOKEN

4. Paste your Hugging Face access token as its value.
5. Enable notebook access for the secret.

In [5]:
hf_token = userdata.get("HF_TOKEN")

if hf_token:
    login(hf_token)
    print("Hugging Face login successful.")
else:
    print("HF_TOKEN was not found in Colab Secrets.")

Hugging Face login successful.


## 5. Model Configuration

We use two open-source models:

### Whisper
Used for:
Audio → Text

### Llama 3.2 3B Instruct
Used for:
Text → Synthetic Meeting / Meeting Minutes

Llama will be loaded using 4-bit quantization so that it can run efficiently
on the Colab GPU.

In [6]:
WHISPER_MODEL = "openai/whisper-medium.en"

LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

print("Whisper:", WHISPER_MODEL)
print("Llama:", LLAMA_MODEL)

Whisper: openai/whisper-medium.en
Llama: meta-llama/Llama-3.2-3B-Instruct


# 6. Real Meeting Mode — Audio to Text

The first part of the system converts meeting audio into a text transcript.

We use the Hugging Face `pipeline()` API with Whisper.

### Workflow

Audio File
↓
Whisper
↓
Transcript

This is an automatic speech recognition task.

In [7]:
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True
)

print("Whisper pipeline loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda


Whisper pipeline loaded successfully.


## 6.1 Load Meeting Audio

For the first test, we will use the Denver meeting audio that was used
in the Week 3 learning material.

The application will eventually allow the user to upload their own audio
through Gradio.

In [8]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
audio_filename = "/content/drive/MyDrive/denver_extract.mp3"

print("Audio file:", audio_filename)
print("File exists:", os.path.exists(audio_filename))

Audio file: /content/drive/MyDrive/denver_extract.mp3
File exists: True


## 6.2 Transcribe the Meeting

Whisper converts the meeting audio into text.

The output will become the input for our Llama meeting-minutes generator.

In [10]:
result = whisper_pipe(audio_filename)

transcription = result["text"]

print(transcription)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the 

In [11]:
open_source_transcription = transcription

print("Transcription generated successfully.")
print("Characters:", len(open_source_transcription))

Transcription generated successfully.
Characters: 12528


# 7. Load Llama 3.2 3B Instruct

Llama will be used for two tasks:

1. Generate synthetic meeting conversations.
2. Convert meeting transcripts into structured meeting minutes.

To reduce GPU memory usage, the model is loaded using 4-bit quantization.

### Quantization

Quantization reduces the numerical precision used to represent model
weights. In this project, we use 4-bit loading through BitsAndBytes.

This allows the 3B parameter model to run more efficiently on a Colab GPU.

In [12]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("4-bit quantization configuration created.")

4-bit quantization configuration created.


## 7.1 Load Tokenizer

The tokenizer converts our text messages into tokens that the Llama model
can process.

We will also use the model's chat template to format system and user
messages correctly.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully.


## 7.2 Load Llama Model

The Llama model is loaded with:

- 4-bit quantization
- Automatic device mapping

The model will use the available GPU.

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    device_map="auto",
    quantization_config=quant_config
)

print("Llama model loaded successfully.")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Llama model loaded successfully.


# 8. Meeting Minutes Generator

The transcript is sent to Llama using a structured chat prompt.

The model will generate:

- Summary
- Attendees
- Location
- Date
- Discussion Points
- Takeaways
- Action Items with Owners

The output will be formatted as Markdown.

def generate_meeting_minutes(transcript):

    system_message = """
You produce minutes of meetings from transcripts.

Return the result in markdown format without code blocks.

Include:
- Summary
- Attendees
- Location
- Date
- Discussion Points
- Takeaways
- Action Items with Owners

Only use information available in the transcript.
Do not invent missing facts.
"""

    user_prompt = f"""
Below is a meeting transcript.

Create structured meeting minutes.

Transcript:
{transcript}
"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=1500
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [23]:
def generate_meeting_minutes(transcript):

    system_message = """
You produce meeting minutes from transcripts.

Return the result in markdown format.

Include only the following sections:

## Summary
## Discussion Points
## Takeaways
## Action Items

For Action Items, include the owner when the transcript clearly identifies one.

Rules:
- Use only information available in the transcript.
- Do not invent names, dates, locations, attendees, decisions, or facts.
- If information is not available, do not create it.
- Keep the summary concise.
- Do not repeat the transcript.
- Return only the meeting minutes.
"""

    user_prompt = f"""
Create structured meeting minutes from the following transcript.

Transcript:
{transcript}
"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Create tokenized chat input with attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True
    )

    # Move all inputs to GPU
    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # Generate the response
    outputs = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    # Get only the newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    # Decode only the assistant's response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

## 8.1 Test Meeting Minutes Generation

We now pass the Whisper transcript to Llama.

### Workflow

Whisper Transcript
↓
Chat Template
↓
Llama 3.2 3B
↓
Meeting Minutes

In [24]:
meeting_minutes = generate_meeting_minutes(
    open_source_transcription
)

print(meeting_minutes)

assistant

## Summary
The Denver City Council meeting of October 9th, 2017, discussed and adopted Proclamation 1127, an observance of the annual Indigenous Peoples Day in the City and County of Denver.

## Discussion Points
- The importance of water in the city's history and the merging of two rivers.
- The recognition of indigenous peoples' contributions to the city's history and culture.
- The celebration of Indigenous Peoples Day as a way to promote inclusivity and respect for all cultures.
- The need to address critical issues affecting the Native American community, such as poverty and lack of access to services.

## Takeaways
- The city's commitment to honoring the cultural and foundational contributions of indigenous people.
- The importance of preserving Native American culture and traditions.
- The need for continued support and recognition of the Native American community's contributions to the city.

## Action Items
- Councilman Lopez to present the proclamation and thank th

# 9. Synthetic Meeting Generator

Synthetic meeting data is artificially generated meeting content.

Instead of uploading an actual meeting, the user provides a topic.

For example:

Topic:
"Planning a university AI hackathon"

The Llama model creates a fictional conversation between participants.

This synthetic transcript can then be passed through the same meeting-minutes
pipeline used for real meetings.

### Workflow

Meeting Topic
↓
Llama
↓
Synthetic Meeting Transcript
↓
Meeting Minutes Generator
↓
Structured Meeting Minutes

def generate_synthetic_meeting(topic):

    system_message = """
You generate realistic but completely fictional meeting transcripts.

Create a short professional meeting conversation between multiple participants.

Include:
- Different participants
- Discussion about the topic
- Different opinions or suggestions
- At least one decision
- At least two action items

Do not include real personal information.
"""

    user_prompt = f"""
Generate a fictional meeting transcript about:

{topic}

Use 3 to 4 participants.
Make the conversation realistic and concise.
"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=1000
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [25]:
def generate_synthetic_meeting(topic):

    system_message = """
You generate realistic but completely fictional meeting conversations.

Create ONLY the conversation between participants.

Requirements:
- Use 3 to 4 fictional participants.
- Give each participant a simple fictional first name.
- Discuss the requested topic.
- Include different opinions or suggestions.
- Include at least one clear decision.
- Include at least two clear action items.
- Keep the conversation concise and professional.
- Do not create a date.
- Do not create a location.
- Do not create attendee biographies.
- Do not include real personal information.
- Do not generate meeting minutes.
- Do not generate a summary.
- Do not add a title outside the conversation.

The output must be only the fictional meeting conversation.
"""

    user_prompt = f"""
Generate a fictional professional meeting conversation about:

{topic}

Use 3 to 4 participants.
Make the conversation realistic and concise.
"""

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Create tokenized chat input
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True
    )

    # Move tensors to GPU
    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only keep newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    # Decode only the assistant response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [26]:
synthetic_meeting = generate_synthetic_meeting(
    "Planning a university AI hackathon"
)

print(synthetic_meeting)

assistant

Ethan: Alright, let's get started. We have a few weeks before the university's AI hackathon, and we need to finalize the plans. Who has ideas on the theme?

Ava: I think we should focus on healthcare. We could have teams build AI-powered diagnostic tools or something like that.

Lucas: I agree, healthcare is a great topic. But I think we should also consider education. We could have teams develop AI-based learning platforms or tools to help students with disabilities.

Ethan: Those are both great ideas. I think we should have a mix of both. What about the prize structure? Should we offer a cash prize or something else?

Ava: I think a cash prize would be more motivating for the teams. But we should also consider offering mentorship opportunities with industry partners.

Lucas: That's a great idea, Ava. I'd be happy to connect some of my contacts in the field. We should also make sure to have a diverse set of teams and judges. We don't want to be seen as biased towards any pa

In [27]:
synthetic_minutes = generate_meeting_minutes(
    synthetic_meeting
)

print(synthetic_minutes)

assistant

## Summary
The meeting discussed finalizing plans for the university's AI hackathon. The theme was decided to be a mix of healthcare and education, with teams building AI-powered diagnostic tools and AI-based learning platforms. The prize structure was also discussed, with a cash prize and mentorship opportunities with industry partners being considered.

## Discussion Points
- Theme: Healthcare and education
- Prize structure: Cash prize and mentorship opportunities
- Diversity: Ensuring a diverse set of teams and judges
- Promotion: Ava to promote the event on social media channels
- Reminder email: Ethan to send a reminder email to the university's student body

## Takeaways
- Finalize plans for the AI hackathon
- Decide on a mix of healthcare and education as the theme
- Consider a cash prize and mentorship opportunities for the prize structure

## Action Items
- Ethan: Send out a reminder email to the university's student body about the hackathon
- Ava: Promote the even

# 10. Meeting Sentiment Analysis

As an additional analysis feature, we use a Hugging Face sentiment-analysis
pipeline on the generated transcript.

This demonstrates how multiple Hugging Face pipelines can be combined
with an open-source LLM workflow.

In [28]:
sentiment_pipe = pipeline(
    "sentiment-analysis"
)

print("Sentiment pipeline loaded.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Sentiment pipeline loaded.


In [30]:
sentiment_result = sentiment_pipe(
    synthetic_meeting[:5000]
)

print(sentiment_result)

[{'label': 'POSITIVE', 'score': 0.9944971203804016}]


# Add zero-shot meeting category

In [31]:
classifier = pipeline(
    "zero-shot-classification"
)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [35]:
categories = [
    "Technology",
    "Education",
    "Business",
    "Project Management",
    "Finance",
    "Other"
]

category_result = classifier(
    synthetic_meeting[:5000],
    categories
)

print("## Meeting Category Analysis\n")
for label, score in zip(category_result['labels'], category_result['scores']):
    print(f"- **{label}**: {score:.2%}")

## Meeting Category Analysis

- **Education**: 65.93%
- **Other**: 15.03%
- **Technology**: 10.32%
- **Project Management**: 3.60%
- **Business**: 2.78%
- **Finance**: 2.34%


# 11. Gradio Application

The final application provides two modes:

### Real Meeting
Upload an audio file and generate meeting minutes.

### Synthetic Meeting
Enter a topic and generate a fictional meeting followed by meeting minutes.

The interface will display:

- Transcript
- Meeting Minutes
- Sentiment
- Meeting Category

In [36]:
def process_real_meeting(audio):

    if audio is None:
        return "Please upload an audio file.", ""

    result = whisper_pipe(audio)

    transcript = result["text"]

    minutes = generate_meeting_minutes(transcript)

    return transcript, minutes

In [38]:
def process_synthetic_meeting(topic):

    if not topic or not topic.strip():
        return "Please enter a meeting topic.", ""

    transcript = generate_synthetic_meeting(topic)

    minutes = generate_meeting_minutes(transcript)

    return transcript, minutes

## 11.1 User Interface

The Gradio interface contains two tabs:

- Real Meeting
- Synthetic Meeting

Each tab has its own input and processing button.

In [40]:
with gr.Blocks(title="Ai Meeting Intelligence") as demo:

    gr.Markdown(
        """
        # 🎙️ AI Meeting Intelligence

        Convert meetings into structured meeting minutes using
        open-source AI models from Hugging Face.
        """
    )

    with gr.Tab("🎧 Real Meeting"):

        audio_input = gr.Audio(
            type="filepath",
            label="Upload Meeting Audio"
        )

        real_button = gr.Button(
            "Analyze Meeting"
        )

        real_transcript = gr.Textbox(
            label="Transcript",
            lines=10
        )

        real_minutes = gr.Markdown(
            label="Meeting Minutes"
        )

        real_button.click(
            fn=process_real_meeting,
            inputs=audio_input,
            outputs=[
                real_transcript,
                real_minutes
            ]
        )

    with gr.Tab("🧪 Synthetic Meeting"):

        topic_input = gr.Textbox(
            label="Meeting Topic",
            placeholder="Example: Planning a university AI hackathon"
        )

        synthetic_button = gr.Button(
            "Generate Meeting"
        )

        synthetic_transcript = gr.Textbox(
            label="Synthetic Meeting Transcript",
            lines=10
        )

        synthetic_minutes = gr.Markdown(
            label="Meeting Minutes"
        )

        synthetic_button.click(
            fn=process_synthetic_meeting,
            inputs=topic_input,
            outputs=[
                synthetic_transcript,
                synthetic_minutes
            ]
        )

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3804ed3cb2a89734ac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


              AI MEETING INTELLIGENCE
                       │
             ┌─────────┴─────────┐
             │                   │
        REAL MEETING        SYNTHETIC MEETING
             │                   │
          Audio                 Topic
             │                   │
          Whisper               Llama
             │                   │
             └─────────┬─────────┘
                       │
                       ▼
                   Transcript
                       │
                       ▼
                 Llama Analysis
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
          Summary   Actions   Takeaways
                       │
                       ▼
                    Gradio